# Train MAP-style predictor from teacher index 22 (layer 23)
Frozen base, canonical Front-to-Back cameras. Only the 197,120-parameter predictor trains. Train/validation never execute decoder blocks. Inputs are read-only; no automatic uploads.

Enable Internet and **two GPUs**. Attach the existing vision cache and the **complete, unpacked train/val teacher export** as Kaggle Inputs. This notebook does not generate targets or raw images. CUDA train batch 32/GPU and greedy test batch 64/GPU still require the real stress checks below.

Training runs for **5 epochs**. Reuse a complete projected cache via PROJECTED_CACHE_INPUT, or leave it empty to build once from the existing raw vision cache. Raw cache is unchanged. Old predictor runs cannot resume into this optimized pipeline.


In [ ]:
from pathlib import Path
import os, subprocess, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
SOURCE = Path('/kaggle/working/Log_Metrics')
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'smolvlm-attention',
                    'https://github.com/huyvanzzz/Log_Metrics.git', str(SOURCE)], check=True)
branch = subprocess.check_output(['git', '-C', str(SOURCE), 'branch', '--show-current'], text=True).strip()
assert branch == 'smolvlm-attention', f'Wrong source branch: {branch}'
assert (SOURCE/'smolvlm_map_predictor/cli.py').is_file(), 'Source is stale: upload/pull the predictor implementation first.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(SOURCE/'smolvlm_map_predictor/requirements.txt')], check=True)
os.environ['PYTHONPATH'] = str(SOURCE)
sys.path.insert(0, str(SOURCE))
assert (SOURCE/'smolvlm_predictor_io/compact.py').is_file(), 'Source is stale: update the optimized predictor code first.'
os.chdir(SOURCE)


## Inputs and resume
Set `TARGETS_INPUT` to the directory containing **both** `train/manifest.json` and `val/manifest.json`. `CACHE_INPUT` can be the cache directory or its enclosing Kaggle Input; leave it empty to use the existing resolver. Prepared data and best base checkpoint download from the pinned public HF repositories when not provided locally.

New run: leave `RESUME_INPUT` and `RESUME_CHECKPOINT` empty. To continue after a stopped session, attach/unpack the previous predictor artifact and set `RESUME_INPUT` to its root (`run.json` + `checkpoints/`). Both latest and best, including their `.rng` directories, are retained. This restores the original recipe, not a fresh run. No `RESUME_TRAINING` flag is used.


In [ ]:
TARGETS_INPUT = '/kaggle/input/YOUR-TEACHER-DATASET/teacher_targets'  # edit: contains train/ and val/
PROJECTED_CACHE_INPUT = ''  # optional: mounted complete projected FP32 cache
PROJECTED_CACHE_DIR = '/kaggle/working/smolvlm_projected_visual_cache'
CACHE_INPUT = ''  # optional: directory containing the existing features.float16.memmap
PREPARED_DIR = ''  # optional: directory containing manifest.json and train/val/test.jsonl
BASE_CHECKPOINT = ''  # optional local finetune_best.pt; requires prepared data and raw or projected cache
OUTPUT = Path('/kaggle/working/smolvlm_map_predictor_index22_optimized_outputs')
RESUME_INPUT = ''  # optional unpacked previous predictor artifact root
RESUME_CHECKPOINT = ''  # optional latest checkpoint already inside OUTPUT, for same-session resume

import json, shutil
import torch, transformers, tokenizers, accelerate
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, 'Enable two CUDA GPUs.'
assert transformers.__version__ == '4.46.3' and tokenizers.__version__ == '0.20.4'
assert accelerate.__version__ == '1.15.0'
print({name: value for name,value in [('torch',torch.__version__),('transformers',transformers.__version__),
      ('accelerate',accelerate.__version__),('GPUs',[torch.cuda.get_device_name(i) for i in range(2)])]})
assert all((Path(TARGETS_INPUT)/split/'manifest.json').is_file() for split in ('train','val')), 'Set TARGETS_INPUT to a complete unpacked export.'
OUTPUT.mkdir(parents=True, exist_ok=True)
if RESUME_INPUT:
    origin = Path(RESUME_INPUT)
    assert origin.resolve() != OUTPUT.resolve()
    assert (origin/'run.json').is_file(), 'Resume artifact must include run.json.'
    if (OUTPUT/'run.json').exists():
        raise RuntimeError('OUTPUT already has a run. Clear RESUME_INPUT to resume it, or choose a fresh OUTPUT.')
    # Copy only small predictor artifacts; never copy targets, cache or base weights.
    for name in ('run.json','training_status.json'):
        if (origin/name).is_file(): shutil.copy2(origin/name, OUTPUT/name)
    for kind in ('latest','best'):
        filename = f'predictor_{kind}.pt'
        assert (origin/'checkpoints'/filename).is_file(), f'Missing {filename}'
        (OUTPUT/'checkpoints').mkdir(exist_ok=True)
        shutil.copy2(origin/'checkpoints'/filename, OUTPUT/'checkpoints'/filename)
        sidecar = origin/'checkpoints'/(filename+'.rng')
        assert sidecar.is_dir(), f'Missing RNG sidecar {sidecar}'
        shutil.copytree(sidecar, OUTPUT/'checkpoints'/sidecar.name)
    if (origin/'debug').is_dir(): shutil.copytree(origin/'debug', OUTPUT/'debug', dirs_exist_ok=True)
    RESUME_CHECKPOINT = str(OUTPUT/'checkpoints/predictor_latest.pt')
if (OUTPUT/'run.json').exists() and not RESUME_CHECKPOINT:
    raise RuntimeError('Run exists. Set RESUME_CHECKPOINT to its latest checkpoint; do not silently restart.')
RUN = OUTPUT/'run.json'
def run(command, *args):
    from smolvlm_predictor_io.progress import run_notebook
    return run_notebook('smolvlm_map_predictor', command, args, OUTPUT, SOURCE)


## Validate inputs once
Checks full cache/target integrity and semantic identity. Workers later use lightweight receipts, per-batch checks and lazy memmaps. No answers enter training prompts. This step may read the full cache once, without loading it all into RAM.


In [ ]:
from smolvlm_topk_benchmark.scoring import metric_preflight
# Detailed preflight output stays in files as well.
from contextlib import redirect_stdout, redirect_stderr
(OUTPUT/'logs').mkdir(exist_ok=True)
with (OUTPUT/'logs'/'metric_preflight.log').open('a') as stream, redirect_stdout(stream), redirect_stderr(stream):
    metric_preflight()
args = ['--config', SOURCE/'configs/smolvlm_map_predictor_index22.json',
        '--targets-input', TARGETS_INPUT, '--output', OUTPUT]
args += ['--projected-cache-input', PROJECTED_CACHE_INPUT, '--projected-cache-dir', PROJECTED_CACHE_DIR]
if CACHE_INPUT: args += ['--cache-input', CACHE_INPUT]
if PREPARED_DIR: args += ['--prepared-dir', PREPARED_DIR]
if BASE_CHECKPOINT: args += ['--base-checkpoint', BASE_CHECKPOINT]
if RESUME_CHECKPOINT: args += ['--resume']
run('validate-inputs', *args)


## Diagnose training on both GPUs
Uses the real loader/prefetch queue, each rank's longest prompts, frozen connector/embeddings, predictor backward and initialized AdamW states. Decoder/vision/LM-head forward calls are forbidden. Failure stops; no automatic batch/precision reduction. This is a separate process and does not write training weights.


In [ ]:
run('diagnose', '--run', RUN)
assert json.loads((OUTPUT/'diagnose.json').read_text())['ok']
print('Diagnose passed. Details: ' + str(OUTPUT/'diagnose.json'))


## Train continuously to 10 completed epochs
FP32, batch 32/GPU, accumulation 1, seed 42. Train uses Accelerate's seeded shuffled sampler with even batches; validation covers each sample exactly once across ranks. Best is selected by sample-mean validation KL. Each completed epoch saves latest/best predictor weights, optimizer/scheduler and per-rank RNG sidecars. An interruption restarts from the most recent complete saved epoch.


In [ ]:
args = ['--run', RUN]
if RESUME_CHECKPOINT:
    run('verify-resume-checkpoint', '--run', RUN, '--predictor', RESUME_CHECKPOINT)
    args += ['--resume', RESUME_CHECKPOINT]
try:
    run('train', *args)
finally:
    # Keep a downloadable recovery artifact even if the child fails or this cell is interrupted.
    # This bundles existing committed checkpoints only; it does not claim the epoch completed.
    if RUN.is_file(): run('bundle', '--run', RUN)


## Greedy QA test with predictor best
Training processes and their worker pools have exited. Two independent GPU workers each stress batch 64 before either starts committing test predictions. Test does not read teacher targets. Only the original 32 selected visual embeddings enter block 0; original RoPE positions and all non-image tokens are preserved. No online teacher, Stage 1, quota or new text processing.


In [ ]:
status = json.loads((OUTPUT/'training_status.json').read_text())
assert status['training_complete'], 'Training is incomplete: resume first; no automatic partial-run QA test.'
BEST = OUTPUT/'checkpoints/predictor_best.pt'
assert BEST.is_file(), 'Restore predictor best along with latest when resuming sessions.'
run('evaluate', '--run', RUN, '--predictor', BEST)
print(json.loads((OUTPUT/'metrics_display.json').read_text()))


## Bundle
Includes predictor checkpoints + RNG, runtime config, logs and predictions/metrics. Excludes frozen base checkpoint, targets and vision cache. No upload/publishing credentials. Keep this artifact to resume; CUDA capacity, speed and quality are not inferred from CPU fixtures.


In [ ]:
run('bundle', '--run', RUN)
from IPython.display import FileLink, display
display(FileLink(str(OUTPUT/'map_predictor_artifacts.zip')))
display(FileLink(str(OUTPUT/'map_predictor_artifacts.sha256.json')))
